# CS781 Phase 6 - Abstractive Summarization - notebook `10`

**Task.** Legal judgment -> abstractive summary. Primary metric BERTScore F1 (roberta-large, the bert-score default for
English); ROUGE-1/ROUGE-L secondary. BERTScore reads at most ~510 roberta tokens (~400 words) of each text.
**Split.** train.json -> fit (95%) + val (5%, fixed, never trained on; picks the epoch by val cross-entropy and the summary
length by val BERTScore F1). test.json is only predicted.
**Decoder models (B, C, D, E):** `[input][separator][summary]`, loss on the summary only (first 576 tokens), greedy
generation with 4-gram blocking. **Inputs** (all published long-document methods):
`extract` = extract-then-abstract (Pilault et al., 2020; sentence classifier trained on ROUGE-oracle labels, 448 tokens),
`lead` = truncation baseline, `chunk` = divide-and-conquer (Shukla et al., 2022 / DANCER: chunk -> chunk summary,
concatenated), `sled` = SLED (Ivgi et al., 2023) for the encoder-decoder A (3,072 tokens in overlapping 512-token windows).
**Knowledge distillation:** D01 (115M, teacher) -> E (58M, student), word-level KD (KL on the teacher's next-token
distribution, T=2, alpha=0.5); E03 is the same student without a teacher.

| GPU | run | technique | input | estimate | question |
|---|---|---|---|---|---|
| GPU0 | `D18_lorma_stage2` | LoRMA r16 | two-stage (Summ^N): D08 chunk summaries -> summary | ~120 min | rewrite all chunk summaries into one summary (stage 2) |
| GPU1 | `B19_lorma_stage2` | LoRMA r16 | two-stage (Summ^N): D08 chunk summaries -> summary | ~100 min | the same stage 2 on B |

**Inputs:** dataset `cs781-p6-code`, `cs781-legal-data` (tokenizer), Phase-1 checkpoint(s) of model B + D, and the OUTPUT of the earlier notebook that saved `models/D08_full_chunk.pt` (Add Input -> Your Work -> that notebook).
Internet ON (Hugging Face data + bert-score's roberta-large), secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
If the Phase-6 data is not on Hugging Face, attach a dataset with `train.json` and `test.json` - it is found automatically.
Expected: ~3.0 h after ~10 min of setup (tokenization, sentence classifier) and a ~10 min smoke test.
**Outputs:** `submissions/<run>.zip` (copied as soon as each run finishes - submit early), `submissions/X00_extractive_baseline.zip`,
`models/<run>.pt`, `bundle_p6_10.zip` (val/test generations, scores, logs), `REPORT_p6_10.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
NB_START = time.time()
W = "/kaggle/working"
WORK, P6DATA, RUNS, MODELS = f"{W}/p6code", "/tmp/p6data", f"{W}/runs", f"{W}/models"
CACHE = "/tmp/p6_cache.npz"
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions", P6DATA):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("summ_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- metric packages (BERTScore F1 with roberta-large = bert-score's default for lang="en"; ROUGE)
subprocess.run("pip install -q bert-score rouge-score", shell=True)

# ---- Phase-6 data: Hugging Face (Phase_6*/...), else any attached train.json / test.json
try:
    from huggingface_hub import hf_hub_download, list_repo_files
    from kaggle_secrets import UserSecretsClient
    REPO = "Exploration-Lab/CS-781-Capstone"
    tok_ = UserSecretsClient().get_secret("HF_TOKEN")
    files = [f for f in list_repo_files(REPO, repo_type="dataset", token=tok_) if f.startswith("Phase_6")]
    print("HF files:", files)
    for f in files:
        dst = os.path.join(P6DATA, f); os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), dst)
except Exception as ex:
    print("Hugging Face download failed:", repr(ex))
def _has(split):
    return any(f.startswith(split) and f.endswith(".json") for _, _, fs in os.walk(P6DATA) for f in fs)
for split, key in (("train", "summary"), ("test", "document")):
    if _has(split):
        continue
    for p in sorted(glob.glob(f"/kaggle/input/**/{split}*.json", recursive=True), key=len):
        try:
            r = json.load(open(p))
            r0 = r[0] if isinstance(r, list) else None
            if r0 and "document" in r0 and (split == "test") == ("summary" not in r0):
                os.makedirs(f"{P6DATA}/{split}", exist_ok=True); shutil.copy(p, f"{P6DATA}/{split}/{split}.json")
                print(f"{split}: using {p}"); break
        except Exception:
            pass
print("data files:", [os.path.join(r, f) for r, _, fs in os.walk(P6DATA) for f in fs])

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
ALL_PT = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
print("checkpoint files found under /kaggle/input:", *ALL_PT, sep="\n  ")
for p in ALL_PT:
    if os.path.basename(p) != "final.pt" or "smoke" in p.lower():
        continue
    m = re.search(r"run[_-]([a-e])(?:[_/-]|$)", p, re.I) or re.search(r"model[_-]?([a-e])(?:[_/-]|$)", p, re.I)
    if not m:
        continue
    k = m.group(1).lower()
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
CKPT.update(CKPT_OVERRIDE if "CKPT_OVERRIDE" in globals() else {})   # e.g. CKPT_OVERRIDE = {"c": "/kaggle/input/.../final.pt"}
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- unit tests; tokenization + oracle extracts + sentence classifier + chunks, once (~5 min); BERTScore model download
r = subprocess.run(f"python tests_summ.py {TOK_ROOT}", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"
dl = subprocess.Popen(["python", "-c", "from bert_score import score; print(score(['a b c'], ['a b d'], lang='en')[2])"],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
import summ_data
t0 = time.time()
INFO = summ_data.build(P6DATA, TOK_ROOT, CACHE)
print(f"cache built in {time.time() - t0:.0f}s:", INFO)
print("BERTScore model check:", dl.communicate()[0][-300:])

code: ['adapt.py', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'models.py', 'ner_data.py', 'peft.py', 'summ_coarse.py', 'summ_data.py', 'summ_model.py', 'summ_train.py', 'tests_summ.py']
HF files: ['Phase_6_Summ/Test/test.json', 'Phase_6_Summ/Train/train.json']


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

data files: ['/tmp/p6data/Phase_6_Summ/Train/train.json', '/tmp/p6data/Phase_6_Summ/Test/test.json']
checkpoint files found under /kaggle/input:
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/best.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/final.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/latest.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/best.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/latest.pt
  /kaggle/input/notebooks/mrinaldevnath/cs781-p6-4/models/B09_full_chunk.pt
  /kaggle/input/notebooks/mrinaldevnath/cs781-p6-4/models/D08_

In [2]:
NB_HOURS = 11.0                                   # Kaggle stops at 12 h; no run starts after 11 h and runs are cut to fit

def cmd(e, out, extra, model_dir):
    args = e["args"].replace("{MODEL_DIR}/", model_dir + ("/smoke_" if smoke_mode[0] else "/"))
    args = re.sub(r"\{CKPT:(\w)\}", lambda m: CKPT[m.group(1)], args)
    if "{COARSE}" in args:                           # two-stage runs: stage-1 summaries written by the stage-1 cell
        args = args.replace("{COARSE}", SMOKE_COARSE if smoke_mode[0] else COARSE_DIR)
    def _saved(m):                                   # a run saved by an earlier notebook, attached as input
        hits = sorted(glob.glob(f"/kaggle/input/**/{m.group(1)}.pt", recursive=True), key=len)
        assert hits, f"{m.group(1)}.pt not found under /kaggle/input - attach the notebook output that contains it"
        return hits[0]
    args = re.sub(r"\{SAVED:(\w+)\}", _saved, args)
    return (f"python summ_train.py --ckpt {CKPT[e['model']]} --tok_root {TOK_ROOT} --cache {CACHE} "
            f"--out {out} --model_dir {model_dir} {args} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
               TOKENIZERS_PARALLELISM="false")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-300:], flush=True)
    if not smoke_mode[0] and os.path.exists(f"{RUNS}/{tag}/submission.zip"):
        shutil.copy(f"{RUNS}/{tag}/submission.zip", f"{W}/submissions/{tag}.zip")     # ready as soon as the run ends
    return p.returncode

smoke_mode = [False]
SMOKE = "--max_docs 12 --max_steps 2 --epochs 1 --micro 2 --accum 1 --out_len 24 --min_new 4 --ch_out 24 --chunk_budget 64 --gen_batch 4 --words 10,0 --sled_in 1200"

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel."""
    smoke_mode[0] = smoke
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" if smoke else "") + e["tag"]
            if smoke:
                extra = SMOKE
            else:
                left = NB_HOURS * 60 - (time.time() - NB_START) / 60
                if left < 30:
                    print(f"[GPU{gpu}] {t}: SKIPPED (notebook time budget used up)", flush=True)
                    status[t] = "skipped"
                    continue
                # training limit: the planned maximum, or what is left minus time for generation and scoring
                extra = f"--max_minutes {max(10, min(e['max_min'], left - 45 - (90 if 'chunk' in e['tag'] else 25))):.0f}"
            try:
                status[t] = run(t, cmd(e, f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
            except Exception as ex:
                print(f"[GPU{gpu}] {t}: NOT STARTED - {ex!r}", flush=True)
                open(f"{W}/logs/{t}.txt", "w").write(f"not started: {ex!r}")
                status[t] = "error"
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Every experiment on 12 documents, 2 steps, short generation (BERTScore included). A failing experiment is dropped
    (its error is printed) instead of stopping the notebook."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    keep = [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]
    # a distillation run needs its teacher: drop it if the teacher was dropped
    tags = {e["tag"] for q in keep for e in q}
    return [[e for e in q if "{MODEL_DIR}/" not in e["args"] or
             re.search(r"\{MODEL_DIR\}/(\S+)\.pt", e["args"]).group(1) in tags] for q in keep]

## Experiments (one queue per GPU)

In [3]:
NAME = '10'
QUEUES = [[{'tag': 'D18_lorma_stage2',
   'model': 'd',
   'tech': 'LoRMA r16',
   'input': 'two-stage (Summ^N): D08 chunk summaries -> summary',
   'args': '--method lorma --rank 16 --alpha 32 --lr 2e-4 --epochs 5 --input stage2 --coarse_dir {COARSE} --in_len 576 --out_len 448',
   'max_min': 264}],
 [{'tag': 'B19_lorma_stage2',
   'model': 'b',
   'tech': 'LoRMA r16',
   'input': 'two-stage (Summ^N): D08 chunk summaries -> summary',
   'args': '--method lorma --rank 16 --alpha 32 --lr 2e-4 --epochs 5 --input stage2 --coarse_dir {COARSE} --in_len 576 --out_len 448',
   'max_min': 220}]]
ALL_QUEUES = [list(q) for q in QUEUES]

## 0. Extractive baseline (CPU, 1 min)

In [4]:
# Extractive baseline (no neural model): the sentence classifier's top sentences in document order, cut to 400 words.
# A reference point for the abstractive runs and a safe early submission (not one of the required experiments).
import numpy as np
from ner_data import load_tokenizer
D_ = summ_data.Data(CACHE); tk_ = load_tokenizer(TOK_ROOT)
def extract_text(split, k, n_tok=900):
    return " ".join(tk_.decode([int(t) for t in D_.doc_input(split, k, "extract", n_tok)]).split())
refs_ = json.load(open(CACHE + ".refs.json"))
vs_ = D_.S["val"]
r_ = np.array([summ_data.rouge(summ_data.cut_words(extract_text("val", k), 400), refs_[vs_.ids[k]]) for k in range(len(vs_))])
print(f"extractive baseline on val: ROUGE-1 {r_[:, 0].mean():.4f} ROUGE-L {r_[:, 1].mean():.4f}")
if "test" in D_.S:
    ts_ = D_.S["test"]
    pred_ = [{"id": ts_.ids[k], "summary": summ_data.cut_words(extract_text("test", k), 400)} for k in range(len(ts_))]
    json.dump(pred_, open(f"{W}/_p.json", "w"), ensure_ascii=False)
    with zipfile.ZipFile(f"{W}/submissions/X00_extractive_baseline.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{W}/_p.json", arcname="prediction.json")
    print("written submissions/X00_extractive_baseline.zip")

extractive baseline on val: ROUGE-1 0.4709 ROUGE-L 0.2521
written submissions/X00_extractive_baseline.zip


## 0b. Summ^N stage 1 (both GPUs, ~45 min)

In [5]:
# Summ^N stage 1 (Zhang et al., ACL 2022): the saved chunk-and-join model D08 summarises every chunk of every document
# (fit / val / test); its concatenated chunk summaries (~576 tokens per document) are the stage-2 input.
# Both GPUs, half of the documents each; a 12-document check first.
hits = sorted(glob.glob("/kaggle/input/**/D08_full_chunk.pt", recursive=True), key=len)
assert hits, "attach the output of notebook 4 (it contains models/D08_full_chunk.pt)"
SAVED_S1 = hits[0]
COARSE_DIR, SMOKE_COARSE = "/tmp/coarse", "/tmp/coarse_smoke"

def stage1(out, extra, shards=2):
    procs = []
    for g in range(shards):
        c = (f"python summ_coarse.py --saved {SAVED_S1} --base_ckpt {CKPT['d']} --tok_root {TOK_ROOT} --cache {CACHE} "
             f"--out_dir {out} --shard {g} --n_shards {shards} {extra}")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        lg = f"{W}/logs/stage1_{os.path.basename(out)}_{g}.txt"
        procs.append((g, lg, subprocess.Popen(c, shell=True, cwd=WORK, env=env, stdout=open(lg, "w"),
                                              stderr=subprocess.STDOUT)))
    for g, lg, p in procs:
        p.wait()
        print(f"[GPU{g}] stage 1 exit {p.returncode}: " + " / ".join(open(lg).read().strip().splitlines()[-3:])[-400:])
        assert p.returncode == 0, open(lg).read()[-3000:]
    for sp in ("fit", "val", "test"):
        merged = {}
        for g in range(shards):
            f = f"{out}/coarse_{sp}_{g}.json"
            if os.path.exists(f):
                merged.update(json.load(open(f)))
        if merged:
            json.dump(merged, open(f"{out}/coarse_{sp}.json", "w"))
        print(f"  {sp}: {len(merged)} documents")

t0 = time.time()
stage1(SMOKE_COARSE, "--max_docs 12")
print(f"stage-1 check OK ({time.time() - t0:.0f}s); full stage 1 ...")
t0 = time.time()
stage1(COARSE_DIR, "")
print(f"stage 1 done in {(time.time() - t0) / 60:.0f} min")
shutil.copytree(COARSE_DIR, f"{W}/coarse", dirs_exist_ok=True)
cv_ = json.load(open(f"{COARSE_DIR}/coarse_val.json"))           # stage 1 alone, for comparison with stage 2
for n_ in (275, 425, 0):
    r_ = np.array([summ_data.rouge(summ_data.cut_words(" ".join(tk_.decode(cv_[i]).split()), n_), refs_[i]) for i in cv_])
    print(f"stage 1 alone on val, {n_ or 'all'} words: ROUGE-1 {r_[:, 0].mean():.4f} ROUGE-L {r_[:, 1].mean():.4f}")

[GPU0] stage 1 exit 0: fit shard 0: 6 docs, 39 chunks, 532 tokens per document, 21s / val shard 0: 6 docs, 62 chunks, 548 tokens per document, 12s / test shard 0: 6 docs, 32 chunks, 537 tokens per document, 16s
[GPU1] stage 1 exit 0: fit shard 1: 6 docs, 96 chunks, 722 tokens per document, 22s / val shard 1: 6 docs, 42 chunks, 483 tokens per document, 19s / test shard 1: 6 docs, 58 chunks, 539 tokens per document, 11s
  fit: 12 documents
  val: 12 documents
  test: 12 documents
stage-1 check OK (86s); full stage 1 ...
[GPU0] stage 1 exit 0: val shard 0: 170 docs, 1609 chunks, 557 tokens per document, 121s / [test]   generated 261/351 / test shard 0: 50 docs, 351 chunks, 517 tokens per document, 40s
[GPU1] stage 1 exit 0: [test]   generated 333/549 / [test]   generated 548/549 / test shard 1: 50 docs, 549 chunks, 570 tokens per document, 58s
  fit: 6688 documents
  val: 340 documents
  test: 100 documents
stage 1 done in 31 min
stage 1 alone on val, 275 words: ROUGE-1 0.4101 ROUGE-L 0.2

## 1. Smoke test

In [6]:
QUEUES = smoke(QUEUES)

[GPU1] smoke_B19_lorma_stage2: exit 0 | [09:49:14] chosen length: untruncated words (bertscore_f1 0.8196) / [09:49:14] done
[GPU0] smoke_D18_lorma_stage2: exit 0 | [09:49:14] chosen length: untruncated words (bertscore_f1 0.8140) / [09:49:14] done
smoke done in 0.5 min: {'smoke_B19_lorma_stage2': 0, 'smoke_D18_lorma_stage2': 0}
SMOKE OK


## 2. Runs

In [7]:
status = run_queues(QUEUES)

[GPU1] B19_lorma_stage2: exit 0 | [11:09:33] chosen length: 275 words (bertscore_f1 0.8424) / [11:09:33] done
[GPU0] D18_lorma_stage2: exit 0 | [11:35:39] chosen length: 275 words (bertscore_f1 0.8447) / [11:35:39] done
runs done in 106.4 min: {'B19_lorma_stage2': 0, 'D18_lorma_stage2': 0}


## 3. Report

In [8]:
def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT p6 {name} ===",
         "run                  | technique                    | input        | val BERTScore F1 | R-1    | R-L    | words | val CE | best epoch | trainable | train min | tok/s  | mem GB | gen s/doc"]
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            lg = f"{W}/logs/{e['tag']}.txt"
            tail = " / ".join(open(lg).read().strip().splitlines()[-3:])[-300:] if os.path.exists(lg) else "not run"
            L.append(f"{e['tag']:20s} | FAILED/SKIPPED: {tail}")
            continue
        s = json.load(open(p)); c = s["chosen"]
        ep = next((h["epoch"] for h in s["history"] if h["step"] == s["best_step"]), "-")
        L.append(f"{e['tag']:20s} | {e['tech'][:28]:28s} | {e['input'][:12]:12s} | {c.get('bertscore_f1', float('nan')):.4f}           | "
                 f"{c['rouge1']:.4f} | {c['rougeL']:.4f} | {c['words'] or 'all':>5} | {s['val_ce']:.4f} | {ep!s:>10} | "
                 f"{s['params_trainable'] / 1e6:7.2f}M | {s['train_minutes']:9.0f} | {s['tok_s']:6,.0f} | {s['peak_mem_gb']:6.1f} | {s['gen_s_per_doc']:.2f}")
        for r in s["val_by_length"]:
            L.append(f"{'':20s}   length {r['words'] or 'all':>4} words (mean {r['mean_words']:.0f}): " +
                     " ".join(f"{k} {r[k]:.4f}" for k in ("bertscore_f1", "rouge1", "rougeL") if k in r))
    bundle = f"{W}/bundle_p6_{name}.zip"              # everything except model weights
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for f in os.listdir(d):
                    z.write(f"{d}/{f}", arcname=f"{e['tag']}/{f}")
            if os.path.exists(f"{W}/logs/{e['tag']}.txt"):
                z.write(f"{W}/logs/{e['tag']}.txt", arcname=f"{e['tag']}/stdout.txt")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_p6_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT p6 10 ===
run                  | technique                    | input        | val BERTScore F1 | R-1    | R-L    | words | val CE | best epoch | trainable | train min | tok/s  | mem GB | gen s/doc
D18_lorma_stage2     | LoRMA r16                    | two-stage (S | 0.8447           | 0.3906 | 0.2222 |   275 | 2.0818 |          2 |    3.33M |        98 |  5,390 |    8.9 | 0.50
                       length  200 words (mean 200): bertscore_f1 0.8428 rouge1 0.3344 rougeL 0.2002
                       length  275 words (mean 274): bertscore_f1 0.8447 rouge1 0.3906 rougeL 0.2222
                       length  350 words (mean 343): bertscore_f1 0.8437 rouge1 0.4239 rougeL 0.2335
                       length  425 words (mean 400): bertscore_f1 0.8421 rouge1 0.4408 rougeL 0.2382
                       length  all words (mean 402): bertscore_f1 0.8420 rouge1 0.4412 rougeL 0.2383
B19_lorma_stage2     | LoRMA r16                    | two-stage (S | 0.8424           | 0.3895 | 0.2184 